# Theory environment comparison

The analysis' own WFT and path-integral code (`tools.compute_n23_ray_variables`, `compute_F_phi_chi`,
`compute_phase_structure_function`, `compute_wft_structure_function`, `compute_vertical_coherence_length`,
`compute_lambda_phi`) is run on four environments with the same geometry (source 2300 m, receiver 1139 m at
153 km), flat 5000 m bottom and GM parameters (ζ₀ = 7.3 m, j* = 3, N₀ = 3 cph, 250 Hz). The point is to
separate what the environment does to the WFT vs path integral gap from what the code does. See
`theory_environments.py` and `reference/phase_structure_function_handoff.md`.

- `munk`: canonical Munk (1974), range independent, analytic c, N and dc_p/dz.
- `fctd`: the analysis environment (cached ray run, hand-picked `EIG_INDEXS`).
- `fctd_range_avg`: FCTD t, SP averaged over 0-153 km, re-derived.
- `nesma_beta0`: nesma_param ocean model at β = 0, SSH of 2023-04-30 on the SIOE-SION geodesic.

Results are cached in `dataproducts/theory_envs/`; pass `force_overwrite=True` to `run_theory` to recompute.

In [ ]:
import warnings

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from n23_fluctuations_ragland import theory_environments as te
from n23_fluctuations_ragland import tools

%config InlineBackend.figure_format = 'retina'

In [ ]:
names = list(te.ENVIRONMENTS)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    th = {name: te.run_theory(name) for name in names}

colors = dict(zip(names, ["C0", "k", "C2", "C3"], strict=True))
ray_ids = tools.DUCTED_RAY_IDS
for name in names:
    print(f"{name:15s}", th[name]["eig_indexs"])

## Environments

Profiles at mid-path (75 km). μ² is ⟨μ²⟩ = ζ₀² (N₀/N) (dc_p/dz)² / c² (the `mu2` of the ray variables).

In [ ]:
zeta0, N0 = 7.3, 3 * 2 * np.pi / 3600
fig, axs = plt.subplots(1, 4, figsize=(13, 5), sharey=True)
for name in names:
    env = th[name]["envi_field"].sel(range=75e3, method="nearest")
    N_rad = env["N"].where(env["N"] > 0) * 2 * np.pi / 3600
    mu2 = zeta0**2 * (N0 / N_rad) * env["dcdz_p"] ** 2 / env["c"] ** 2
    for ax, var in zip(axs, [env["c"], env["N"], env["dcdz_p"], mu2], strict=True):
        ax.plot(var, env.depth, color=colors[name], label=name)
axs[3].set_xscale("log")
for ax, label in zip(axs, ["c [m/s]", "N [cph]", "dc$_p$/dz [1/s]", r"$\langle\mu^2\rangle$"], strict=True):
    ax.set_xlabel(label)
axs[0].set_ylim(5000, 0)
axs[0].set_ylabel("depth [m]")
axs[0].legend()

In [ ]:
# range dependence: sound speed sections and N0B (N integrated to the bottom) along the path
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for name in names:
    env = th[name]["envi_field"].sel(range=slice(0, 153e3))
    N_rad = env["N"].where(env["N"] > 0, 0) * 2 * np.pi / 3600
    N0B = N_rad.where(N_rad.depth <= 5000, 0).integrate("depth")
    axs[0].plot(N0B.range / 1e3, N0B, color=colors[name], label=name)
    c_axis = env["c"].idxmin("depth")
    axs[1].plot(c_axis.range / 1e3, c_axis, color=colors[name])
axs[0].set_ylabel("$N_0B$ [rad m/s]")
axs[1].set_ylabel("sound channel axis [m]")
axs[1].invert_yaxis()
for ax in axs:
    ax.set_xlabel("range [km]")
axs[0].legend()

In [ ]:
# SSH along the nesma transect
env = th["nesma_beta0"]["envi_field"]
if "ssh" in env.coords:
    ssh = env["ssh"].sel(range=slice(0, 200e3))
    plt.plot(ssh.range / 1e3, ssh)
    plt.axvline(153, color="k", ls=":")
    plt.xlabel("range from SIOE [km]")
    plt.ylabel("ADT [m]")

## Eigenrays and ray variables

Only `-6` is a refracted eigenray in canonical Munk at this geometry; the `+6`/`-7` families end between
2300 and 3200 m at 153 km.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True)
for ax, ray_id in zip(axs.flat, ray_ids, strict=True):
    for name in names:
        for n, rv in enumerate(th[name]["rvs"].get(ray_id, [])):
            ax.plot(rv["rs"] / 1e3, -rv["zs"], color=colors[name], lw=1, label=name if n == 0 else None)
    ax.set_title(ray_id)
axs[0, 0].invert_yaxis()
axs[0, 0].legend()
for ax in axs[1]:
    ax.set_xlabel("range [km]")
for ax in axs[:, 0]:
    ax.set_ylabel("depth [m]")

In [ ]:
variables = ["mu2", "L_p", "Rf2", "RT_zeta_1", "J", "N0B_ray"]
for ray_id in ray_ids:
    fig, axs = plt.subplots(1, len(variables), figsize=(18, 3), sharex=True)
    for name in names:
        for n, rv in enumerate(th[name]["rvs"].get(ray_id, [])):
            for ax, var in zip(axs, variables, strict=True):
                ax.plot(rv["rs"] / 1e3, np.asarray(rv[var]), color=colors[name], lw=1,
                        label=name if n == 0 else None)
    for ax, var in zip(axs, variables, strict=True):
        ax.set_title(var)
        ax.set_xlabel("range [km]")
    axs[0].set_yscale("log")
    axs[0].set_ylabel(ray_id)
    axs[-1].legend(fontsize=7)

## Spectra and structure functions

F per cycle/m (Eq. 6.52, local straight ray). Solid: path integral D (Eq. 7.50, with ξ₁). Dashed: WFT D_φ.
Dotted: WFT D_φ + D_χ, which equals the path integral with ξ₁ = 1 apart from the continuum vs modal form.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, ray_id in zip(axs.flat, ray_ids, strict=True):
    for name in names:
        m = th[name]["m"]
        for n, (Fp, Fc) in enumerate(zip(th[name]["F_phi"].get(ray_id, []), th[name]["F_chi"].get(ray_id, []), strict=True)):
            ax.loglog(m, Fp, color=colors[name], lw=1, label=name if n == 0 else None)
            ax.loglog(m, Fc, color=colors[name], lw=1, ls="--")
    ax.set_title(f"{ray_id}  (solid $F_\\phi$, dashed $F_\\chi$)")
    ax.axvspan(1 / 252, 1 / 18, color="0.9", zorder=0)  # array band
axs[0, 0].set_ylim(1e-3, 1e6)
axs[0, 0].legend()
for ax in axs[1]:
    ax.set_xlabel("m [cpm]")

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, ray_id in zip(axs.flat, ray_ids, strict=True):
    for name in names:
        dz = th[name]["deltaz_r"]
        for n in range(len(th[name]["D_PI"].get(ray_id, []))):
            lab = name if n == 0 else None
            ax.loglog(dz, th[name]["D_PI"][ray_id][n], color=colors[name], label=lab)
            ax.loglog(dz, th[name]["D_wft_phi"][ray_id][n], color=colors[name], ls="--")
            ax.loglog(dz, th[name]["D_wft_phi"][ray_id][n] + th[name]["D_wft_chi"][ray_id][n],
                      color=colors[name], ls=":")
    ax.set_title(ray_id)
    ax.set_xlim(8, 250)
axs[0, 0].set_ylim(1e-3, 1e2)
axs[0, 0].legend()
for ax in axs[1]:
    ax.set_xlabel(r"$\Delta z$ [m]")
for ax in axs[:, 0]:
    ax.set_ylabel("D [rad$^2$]")

## Summary

Same layout as `reference/phase_structure_function_wft_vs_path_integral.md` §5, per environment and micro-ray.

In [ ]:
rows = []
for name in names:
    dz = th[name]["deltaz_r"]
    for ray_id, D_PI in th[name]["D_PI"].items():
        for n in range(len(D_PI)):
            for target in (27, 108):
                i = int(np.argmin(abs(dz - target)))
                Dp, Dc = th[name]["D_wft_phi"][ray_id][n][i], th[name]["D_wft_chi"][ray_id][n][i]
                rows.append({
                    "env": name, "ray": ray_id, "micro": n, "dz": round(dz[i], 1),
                    "PI": D_PI[n][i], "WFT phi": Dp, "WFT phi+chi": Dp + Dc,
                    "PI / WFT phi": D_PI[n][i] / Dp,
                    "z0 [m]": float(th[name]["z0"][ray_id][n]),
                    "Phi2": float(th[name]["Phi2"][ray_id][n]),
                    "Lambda": float(th[name]["Lambda"][ray_id][n]),
                })
summary = pd.DataFrame(rows).set_index(["ray", "dz", "env", "micro"]).sort_index()
summary.round(3)

## Check: nesma β = 0 against the inversion's ocean model

`nesma_beta0_environment` rebuilds the β = 0 state without the horizontal basis. Compare it with
`n23_inversion_ragland`'s model on the same transect (needs the workspace install; not a dependency).

In [ ]:
try:
    from n23_inversion_ragland.observations.run import _build_ocean_model
    from n23_inversion_ragland.transect import resolve_transect_kwargs
except ImportError as err:
    print("n23_inversion_ragland not installed:", err)
else:
    om = _build_ocean_model(
        M=80, N=10, x_max=200e3, z_max=6000.0, ssh_time="2023-04-30", stationary_horizontal=False,
        stationary_ell=15e3, knots_zeta=None, knots_ell=(50e3, 30e3, 20e3, 10e3, 20e3, 30e3, 50e3),
        transect_kwargs=resolve_transect_kwargs(),
    )
    dT, dS, _ = om.forward(np.zeros((om.M, om.N)))
    T_inv = np.asarray(om.mu_T)[:, None] + np.asarray(dT)
    env = th["nesma_beta0"]["envi_field"].sel(range=slice(0, 200e3))
    T_here = env["t"].interp(depth=np.asarray(om.z_grid)).values
    print("max |T - T_inversion| [degC]:", np.nanmax(np.abs(T_here - T_inv)))